Stage 1-2: train and explain

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42
DATA_PATH = Path("diabetes_012_health_indicators_BRFSS2015.csv")
ARTIFACTS = Path("artifacts_notebook")
MODEL_PATH = ARTIFACTS / "diabetes_risk_random_forest.joblib"
TARGET = "Diabetes_012"
FEATURES = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
EXPECTED_CLASSES = np.array([0, 1, 2])
RISK_LABELS = {0: "Low", 1: "Medium (prediabetes)", 2: "High (diabetes)"}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")
source_df = pd.read_csv(DATA_PATH)
missing = set(FEATURES + [TARGET]) - set(source_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df = source_df[FEATURES + [TARGET]].copy()
df[FEATURES + [TARGET]] = df[FEATURES + [TARGET]].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=FEATURES + [TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)
df = df[df[TARGET].isin(EXPECTED_CLASSES)].copy()
X = df[FEATURES].astype(float)
y = df[TARGET].astype(int)

profile_groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
overall_distribution = y.value_counts(normalize=True).reindex(EXPECTED_CLASSES, fill_value=0)
def split_score(pair):
    _, test_position = pair
    test_distribution = y.iloc[test_position].value_counts(normalize=True).reindex(EXPECTED_CLASSES, fill_value=0)
    return abs(len(test_position) / len(y) - 0.20) + float((test_distribution - overall_distribution).abs().sum())
train_position, test_position = min(splitter.split(X, y, groups=profile_groups), key=split_score)
X_train, X_test = X.iloc[train_position].copy(), X.iloc[test_position].copy()
y_train, y_test = y.iloc[train_position].copy(), y.iloc[test_position].copy()

ARTIFACTS.mkdir(parents=True, exist_ok=True)
if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    print(f"Loaded existing model: {MODEL_PATH}")
else:
    model = RandomForestClassifier(
        n_estimators=400, class_weight="balanced_subsample", min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    joblib.dump(model, MODEL_PATH, compress=3)
    print(f"Trained and saved model: {MODEL_PATH}")
if list(getattr(model, "feature_names_in_", FEATURES)) != FEATURES:
    raise RuntimeError("The model feature contract does not match the expected 21 BRFSS features.")

In [ ]:
import shap

# Use a reproducible BRFSS test-set row; never embed or auto-load a local patient file.
example_patient = X_test.iloc[[0]].copy()

patient_prediction = int(model.predict(example_patient)[0])
class_position = int(np.where(model.classes_ == patient_prediction)[0][0])
explainer = shap.TreeExplainer(model)
raw_shap = explainer.shap_values(example_patient)
if isinstance(raw_shap, list):
    patient_shap = np.asarray(raw_shap[class_position], dtype=float)[0]
else:
    shap_array = np.asarray(raw_shap, dtype=float)
    if shap_array.shape == (1, len(FEATURES), len(model.classes_)):
        patient_shap = shap_array[0, :, class_position]
    elif shap_array.shape == (len(model.classes_), 1, len(FEATURES)):
        patient_shap = shap_array[class_position, 0, :]
    else:
        raise RuntimeError(f"Unsupported SHAP output shape: {shap_array.shape}")
contributions = [
    {"feature": feature, "value": float(example_patient.iloc[0][feature]), "shap_value": float(shap_value)}
    for feature, shap_value in zip(FEATURES, patient_shap)
]
print(f"Explained one patient for predicted class {patient_prediction} with {len(contributions)} contributions.")

Stage 3: what-if simulation

In [ ]:
from stage3 import DiabetesDigitalTwin, smpl_twin_descriptor

twin = DiabetesDigitalTwin(MODEL_PATH)
baseline = {feature: float(example_patient.iloc[0][feature]) for feature in FEATURES}
scenario = baseline.copy()
scenario["PhysActivity"] = 1.0 - baseline["PhysActivity"]
scenario["BMI"] = max(10.0, baseline["BMI"] - 5.0)
scenario["Fruits"] = 1.0 - baseline["Fruits"]

stage3_result = twin.simulate(baseline, scenario)
stage3_result["important_limitation"] = (
    "This is model re-prediction after manual input edits; it is not a causal estimate or treatment recommendation."
)
baseline_prediction = stage3_result["baseline"]
stage3_twin = smpl_twin_descriptor(baseline["BMI"], baseline_prediction["high_risk_probability"])
print(f"Baseline high-risk probability: {stage3_result['baseline']['high_risk_probability']:.4f}")
print(f"Scenario high-risk probability: {stage3_result['scenario']['high_risk_probability']:.4f}")
print("Changed features:", json.dumps(stage3_result["changes"], indent=2))
(ARTIFACTS / "stage3_case_study.json").write_text(json.dumps(stage3_result, indent=2), encoding="utf-8")

Stage 4: knowledge graph

In [ ]:
from knowledge_graph import PatientKnowledgeGraph

class OfflineDriver:
    def session(self):
        raise RuntimeError("Offline notebook mode: Neo4j connection intentionally disabled.")

high_risk_contributions = twin.explain(baseline, max_factors=None, class_id=2)
knowledge_graph_builder = PatientKnowledgeGraph(driver=OfflineDriver())
knowledge_graph = knowledge_graph_builder.explain(
    profile=baseline,
    prediction=baseline_prediction,
    contributions=high_risk_contributions,
    model_name="diabetes_risk_random_forest",
)
digital_twin_graph = {"nodes": knowledge_graph["nodes"], "edges": knowledge_graph["edges"]}
print(f"Nodes: {len(digital_twin_graph['nodes'])}")
print(f"Edges: {len(digital_twin_graph['edges'])}")
(ARTIFACTS / "knowledge_graph.json").write_text(json.dumps(digital_twin_graph, indent=2), encoding="utf-8")

Stage 4: LLM guidance (optional)

In [ ]:
import os
import socket
from ollama_recommendations import OllamaRecommendationError, generate_local_guidance

previous_ollama_timeout = os.environ.get("OLLAMA_TIMEOUT_SECONDS")
try:
    # A short preflight prevents the optional local service from stalling the pipeline.
    with socket.create_connection(("127.0.0.1", 11434), timeout=0.5):
        pass
    os.environ["OLLAMA_TIMEOUT_SECONDS"] = "10"
    guidance = generate_local_guidance(1, baseline_prediction, knowledge_graph, stage3_twin)
    print(json.dumps({"guidance": guidance}, indent=2))
except (OllamaRecommendationError, ConnectionError, OSError, TimeoutError):
    ranked_positive = sorted((item for item in contributions if item["shap_value"] > 0), key=lambda item: item["shap_value"], reverse=True)
    ranked_negative = sorted((item for item in contributions if item["shap_value"] < 0), key=lambda item: item["shap_value"])
    topic_by_feature = {
        "HighBP": "blood_pressure", "HighChol": "cholesterol", "PhysActivity": "physical_activity",
        "Fruits": "nutrition", "Veggies": "nutrition", "Smoker": "smoking", "DiffWalk": "mobility",
        "GenHlth": "general_health", "AnyHealthcare": "healthcare_access", "NoDocbcCost": "healthcare_access",
        "MentHlth": "wellbeing", "PhysHlth": "wellbeing", "HvyAlcoholConsump": "alcohol",
    }
    selected = ranked_positive[:3] + ranked_negative[:2]
    discussion_topics = list(dict.fromkeys(topic_by_feature[item["feature"]] for item in selected if item["feature"] in topic_by_feature))[:3]
    sample_output = {
        "supporting_features": [item["feature"] for item in ranked_positive[:3]],
        "opposing_features": [item["feature"] for item in ranked_negative[:2]],
        "discussion_topics": discussion_topics,
    }
    print("SAMPLE OUTPUT — Ollama not running:")
    print(json.dumps(sample_output, indent=2))
finally:
    if previous_ollama_timeout is None:
        os.environ.pop("OLLAMA_TIMEOUT_SECONDS", None)
    else:
        os.environ["OLLAMA_TIMEOUT_SECONDS"] = previous_ollama_timeout

Launch dashboard

In [ ]:
import json
import subprocess
import time
try:
    import requests
except ImportError:
    from pip._vendor import requests

repo_root = Path.cwd().resolve()
compose_base = ["docker", "compose"]

def run_command(args, timeout):
    try:
        return subprocess.run(
            args, cwd=repo_root, capture_output=True, text=True, timeout=timeout
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        print(f"Command failed: {' '.join(args)}")
        print(str(exc))
        return None

docker_check = run_command(compose_base + ["version"], timeout=15)
docker_available = docker_check is not None and docker_check.returncode == 0
if not docker_available:
    if docker_check is not None and docker_check.stderr.strip():
        print(docker_check.stderr.strip())
    print("Docker not available — install Docker Desktop and re-run this cell")

if docker_available:
    neo4j_up = run_command(compose_base + ["up", "-d", "neo4j"], timeout=120)
    neo4j_started = neo4j_up is not None and neo4j_up.returncode == 0
    if not neo4j_started:
        print("Failed to start Neo4j.")
        if neo4j_up is not None:
            print(neo4j_up.stderr.strip() or "(no stderr output)")

    neo4j_healthy = False
    if neo4j_started:
        print("Waiting for Neo4j to become healthy...")
        health_deadline = time.monotonic() + 60.0
        attempt = 0
        while time.monotonic() < health_deadline:
            attempt += 1
            health_result = run_command(
                compose_base + ["ps", "neo4j", "--format", "json"], timeout=15
            )
            if health_result is None:
                break
            if health_result.returncode != 0:
                print(health_result.stderr.strip() or "Failed to inspect Neo4j health.")
            else:
                try:
                    health_payload = json.loads(health_result.stdout or "[]")
                    health_rows = health_payload if isinstance(health_payload, list) else [health_payload]
                    neo4j_healthy = any(
                        str(row.get("Health", "")).lower() == "healthy" for row in health_rows
                    )
                except (json.JSONDecodeError, AttributeError):
                    neo4j_healthy = False
            if neo4j_healthy:
                print("Neo4j is healthy.")
                break
            print(f"Neo4j not healthy yet (attempt {attempt}); retrying...")
            time.sleep(5)
        if not neo4j_healthy:
            print("Neo4j did not become healthy within 60 seconds.")

    dashboard_started = False
    if neo4j_healthy:
        dashboard_ps = run_command(
            compose_base + ["ps", "dashboard", "--status", "running", "--format", "json"],
            timeout=15,
        )
        dashboard_running = (
            dashboard_ps is not None
            and dashboard_ps.returncode == 0
            and bool(dashboard_ps.stdout.strip())
        )
        if dashboard_ps is not None and dashboard_ps.returncode != 0:
            print(dashboard_ps.stderr.strip() or "Failed to inspect dashboard status.")

        if dashboard_running:
            print("Dashboard is already running; restarting it to load fresh artifacts...")
            dashboard_start = run_command(compose_base + ["restart", "dashboard"], timeout=120)
        else:
            image_check = run_command(
                ["docker", "images", "-q", "training3-dashboard:latest"], timeout=15
            )
            image_exists = (
                image_check is not None
                and image_check.returncode == 0
                and bool(image_check.stdout.strip())
            )
            if image_check is not None and image_check.returncode != 0:
                print(image_check.stderr.strip() or "Failed to inspect dashboard image.")
            dashboard_command = compose_base + ["up", "-d"]
            if not image_exists:
                print("Dashboard image not found; building it once...")
                dashboard_command.append("--build")
            dashboard_command.append("dashboard")
            dashboard_start = run_command(dashboard_command, timeout=600)

        dashboard_started = dashboard_start is not None and dashboard_start.returncode == 0
        if not dashboard_started:
            print("Failed to start or restart the dashboard.")
            if dashboard_start is not None:
                print(dashboard_start.stderr.strip() or "(no stderr output)")

    if dashboard_started:
        dashboard_url = "http://127.0.0.1:5000"
        dashboard_deadline = time.monotonic() + 30.0
        dashboard_ready = False
        print("Waiting for the Docker dashboard to respond...")
        while time.monotonic() < dashboard_deadline:
            try:
                response = requests.get(dashboard_url, timeout=1.0)
                if response.status_code < 500:
                    dashboard_ready = True
                    break
            except requests.RequestException:
                pass
            time.sleep(1)

        if dashboard_ready:
            print("Dashboard running (Docker): http://127.0.0.1:5000")
        else:
            print("Dashboard did not respond within 30 seconds. Recent logs:")
            dashboard_logs = run_command(
                compose_base + ["logs", "--tail=50", "dashboard"], timeout=30
            )
            if dashboard_logs is not None:
                if dashboard_logs.stdout.strip():
                    print(dashboard_logs.stdout.strip())
                if dashboard_logs.stderr.strip():
                    print(dashboard_logs.stderr.strip())

Run `docker compose down` in a terminal to stop the dashboard and Neo4j containers when finished.